# Probe: Serverless GPU AI Runtime (env5) + pycolmap-cuda12

Throwaway feasibility spike for the orthomosaic GPU/dense milestone. Validates that the
`photogrammetry_gpu_env5` extra installs on Serverless GPU env5, that CUDA is live, and
that pycolmap-cuda12 exposes the dense MVS API (`patch_match_stereo`/`stereo_fusion`/
`import_undistort`). Results are returned via `dbutils.notebook.exit(...)` (the Jobs API
surfaces that as notebook_output; serverless stdout is not exposed).

In [ ]:
%pip install "geobrix[light_env5,photogrammetry_gpu_env5] @ file:///Volumes/geospatial_docs/gdal_artifacts/noble/geobrix/geobrix-0.5.2-py3-none-any.whl"
%restart_python

In [ ]:
import json, subprocess, sys, platform

out = {"python": platform.python_version()}

# 1) GPU visibility (confirms accelerator model: A10 / H100)
try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=60,
    )
    out["nvidia_smi_rc"] = smi.returncode
    out["gpu"] = (smi.stdout or smi.stderr).strip()
except Exception as e:  # noqa: BLE001
    out["gpu"] = f"ERR: {e}"

# 2) CUDA toolkit version
try:
    nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True, timeout=30)
    out["nvcc"] = (nvcc.stdout or nvcc.stderr).strip().splitlines()[-1] if nvcc.returncode == 0 else f"rc={nvcc.returncode}"
except Exception as e:  # noqa: BLE001
    out["nvcc"] = f"ERR: {e}"

# 3) pycolmap-cuda12 import + version + dense-API surface
try:
    import pycolmap
    out["pycolmap_version"] = getattr(pycolmap, "__version__", "?")
    dense_syms = sorted(
        x for x in dir(pycolmap)
        if any(k in x.lower() for k in ("patch", "stereo", "fusion", "undistort", "dense", "mvs"))
    )
    out["pycolmap_dense_api"] = dense_syms
    out["has_patch_match_stereo"] = hasattr(pycolmap, "patch_match_stereo")
    out["has_stereo_fusion"] = hasattr(pycolmap, "stereo_fusion")
    out["has_import_undistort"] = any(hasattr(pycolmap, n) for n in ("import_undistort", "undistort_images"))
    # Does pycolmap report CUDA support?
    out["pycolmap_has_cuda"] = bool(getattr(pycolmap, "has_cuda", None))
except Exception as e:  # noqa: BLE001
    out["pycolmap_error"] = f"{type(e).__name__}: {e}"

# 4) geobrix light tier import (sanity: wheel installed OK on GPU env)
try:
    import databricks.labs.gbx  # noqa: F401
    from databricks.labs.gbx.pyrx import functions as _rx  # noqa: F401
    out["geobrix_import"] = "ok"
except Exception as e:  # noqa: BLE001
    out["geobrix_import"] = f"ERR: {type(e).__name__}: {e}"

print(json.dumps(out, indent=2))
dbutils.notebook.exit(json.dumps(out))